# Tensor Operations

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.02 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/09_tensor_operations.ipynb)

---

## 🎯 Learning Objective

Extend vectors and matrices to tensors, and learn Einstein summation (einsum) notation as the universal language for the batched, multi-dimensional operations inside neural networks.

---

## 📐 Theory

A **tensor** generalizes scalars (rank 0), vectors (rank 1), and matrices (rank 2) to
arbitrarily many indices. A rank-3 tensor $T_{ijk}$ is naturally visualized as a stack of
matrices; a batch of RGB images is a rank-4 tensor indexed by (batch, channel, height, width).
Each index position is called a **mode** or **axis**.

### Slicing and contraction

Fixing some indices and letting others vary is **slicing**: $T_{i,:,:}$ extracts the $i$-th
matrix "sheet." **Contraction** sums over a shared index between two tensors — matrix
multiplication is the simplest example, contracting the shared index $j$ in $C_{ik} = \sum_j
A_{ij}B_{jk}$.

### Einstein summation (einsum) notation

Einsum notation makes contractions explicit and compact by writing only the indices, with any
index that's *repeated but not in the output* implicitly summed over:

| Einsum string | Operation |
|---|---|
| `'ij,jk->ik'` | matrix multiplication |
| `'ij->ji'` | transpose |
| `'ii->'` | trace (sum of diagonal) |
| `'i,i->'` | dot product |
| `'i,j->ij'` | outer product |
| `'bij,bjk->bik'` | **batched** matrix multiplication (the `b` index is kept, not summed) |

This single notation subsumes dot products, matrix multiplication, batched operations, and
outer products — which is exactly why `torch.einsum`/`np.einsum` shows up everywhere in
real model code instead of chains of `reshape`/`transpose`/`matmul`.

### Outer product

The outer product $\mathbf{u}\otimes\mathbf{v}$ of an $m$-vector and $n$-vector produces an
$m\times n$ **rank-1** matrix: $(\mathbf{u}\otimes\mathbf{v})_{ij} = u_i v_j$. Sums of a few
outer products are exactly how we built low-rank matrices in
[Notebook 01.02](02_matrix_operations.ipynb) and [01.07](07_SVD_and_PCA.ipynb).

---

## 👁️ Visual Intuition

A rank-3 tensor is a stack of matrices — slicing along the first axis peels off one matrix
"sheet" at a time.

In [ ]:
# Setup + visualizing a rank-3 tensor as a stack of 2D slices
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

rng = np.random.default_rng(0)
T = rng.standard_normal((4, 5, 5))   # a rank-3 tensor: 4 "sheets", each a 5x5 matrix

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for i, ax in enumerate(axes):
    im = ax.imshow(T[i], cmap="coolwarm", vmin=T.min(), vmax=T.max())
    ax.set_title(f"T[{i}, :, :]  (slice {i})")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f"Tensor T has shape {T.shape} -- a stack of {T.shape[0]} matrices, each {T.shape[1]}x{T.shape[2]}")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We implement matrix multiplication, batched matrix multiplication, and outer products first
as explicit loops (making the summed/free indices completely explicit), then show the
one-line `einsum` equivalent for each.

In [ ]:
# Explicit-loop tensor contractions vs. their einsum equivalents
import numpy as np

rng = np.random.default_rng(0)

# --- Plain matrix multiply: C_ik = sum_j A_ij B_jk ---
A = rng.standard_normal((3, 4))
B = rng.standard_normal((4, 5))
C_loop = np.zeros((3, 5))
for i in range(3):
    for k in range(5):
        C_loop[i, k] = sum(A[i, j] * B[j, k] for j in range(4))
C_einsum = np.einsum('ij,jk->ik', A, B)
print("Matmul loop matches einsum('ij,jk->ik'):", np.allclose(C_loop, C_einsum))
print("Both match A @ B:                       ", np.allclose(C_loop, A @ B))

# --- Batched matrix multiply: C_bik = sum_j A_bij B_bjk  (batch index b is NOT summed) ---
batch = 2
Ab = rng.standard_normal((batch, 3, 4))
Bb = rng.standard_normal((batch, 4, 5))
Cb_loop = np.zeros((batch, 3, 5))
for b in range(batch):
    for i in range(3):
        for k in range(5):
            Cb_loop[b, i, k] = sum(Ab[b, i, j] * Bb[b, j, k] for j in range(4))
Cb_einsum = np.einsum('bij,bjk->bik', Ab, Bb)
print("\nBatched matmul loop matches einsum('bij,bjk->bik'):", np.allclose(Cb_loop, Cb_einsum))

# --- Outer product: (u ox v)_ij = u_i v_j ---
u, v = rng.standard_normal(3), rng.standard_normal(4)
outer_loop = np.array([[u[i] * v[j] for j in range(4)] for i in range(3)])
outer_einsum = np.einsum('i,j->ij', u, v)
print("\nOuter product loop matches einsum('i,j->ij'):", np.allclose(outer_loop, outer_einsum))
print("Both match np.outer:                          ", np.allclose(outer_loop, np.outer(u, v)))

## 🤖 Why This Matters for AI

**Multi-head attention** splits queries/keys/values into `n_heads` independent chunks, computes
attention scores per head in parallel, then concatenates the results. This is naturally a
rank-4 tensor operation: shape `(batch, heads, seq_len, head_dim)`. The einsum string
`'bhid,bhjd->bhij'` computes every head's attention scores for every batch element in one call
— no manual loop over batches or heads required.

In [ ]:
# Multi-head attention SCORES computed with a single einsum call over (batch, heads, seq, dim)
import numpy as np

rng = np.random.default_rng(0)
batch, n_heads, seq_len, head_dim = 2, 4, 6, 8

Q = rng.standard_normal((batch, n_heads, seq_len, head_dim))
K = rng.standard_normal((batch, n_heads, seq_len, head_dim))

# scores[b,h,i,j] = sum_d Q[b,h,i,d] * K[b,h,j,d]  -- every (query i, key j) pair, per batch & head
scores_einsum = np.einsum('bhid,bhjd->bhij', Q, K) / np.sqrt(head_dim)

# Explicit loop version, to make the contraction completely unambiguous
scores_loop = np.zeros((batch, n_heads, seq_len, seq_len))
for b in range(batch):
    for h in range(n_heads):
        scores_loop[b, h] = (Q[b, h] @ K[b, h].T) / np.sqrt(head_dim)

print("Q shape:", Q.shape, " K shape:", K.shape)
print("Attention scores shape:", scores_einsum.shape, "  (batch, heads, query_pos, key_pos)")
print("einsum matches explicit per-batch-per-head loop:", np.allclose(scores_einsum, scores_loop))
print("\nOne einsum call replaced", batch * n_heads, "separate matrix multiplications.")

## 🏋️ Exercises

### Warm-up
1. Given vectors $u=(1,2,3)$ and $v=(4,5)$, compute their outer product by hand as a $3\times2$ matrix, then verify with `np.einsum('i,j->ij', u, v)`.

### Practice
2. Use `np.einsum('ii->', A)` to compute the trace of a random $5\times5$ matrix $A$ (sum of diagonal entries) and verify it matches `np.trace(A)`. Then use `np.einsum('ij->ji', A)` to transpose it and verify against `A.T`.

### Challenge
3. Extend the multi-head attention example: after computing `scores_einsum`, apply softmax along the last axis (key positions) to get attention WEIGHTS, then use a second einsum call `'bhij,bhjd->bhid'` to combine the weights with a value tensor `V` of shape `(batch, n_heads, seq_len, head_dim)`, producing the attended output. Confirm the output shape matches `Q`'s shape, and verify your einsum against an explicit per-batch-per-head loop.

---

## 📚 Further Reading
- [einsum is all you need (blog post with visual explanations)](https://rockt.github.io/2018/04/30/einsum)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)

---

*Next notebook: [Linear Algebra in Transformers](10_linear_algebra_in_transformers.ipynb)*